# Step 04 — v2 Analysis & Visualization

**All new v2 charts:**
1. Class balance
2. Confusion matrix (annotated)
3. Per-class Precision / Recall / F1
4. Training history + LR schedule
5. **UMAP / t-SNE embedding clusters** ← new
6. **Frame strip comparison** (per-class, top-down view) ← new
7. **YOLO person detection grid** ← new
8. **Per-person × per-class accuracy heatmap** ← new
9. **Confidence calibration curve** ← new
10. Top confusion pairs bar chart ← new
11. **3-view side-by-side comparison** ← new
12. Backbone leaderboard
13. PCA outliers

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image, Markdown

In [ ]:
from lib.paths import CHECKPOINTS_DIR, OUTPUTS_DIR
from lib.har_analysis import run_v2_analysis, eval_checkpoint
from lib.inhard import resolve_training_clips
from lib.paths import find_inhard_root
from lib.constants import BLOCKED_ACTIONS, BACKBONE_VJEPA, BACKBONE_DINOV2

# Find best available checkpoint
ckpts = sorted(CHECKPOINTS_DIR.glob('har_vjepa_*.pt'))
if not ckpts:
    print('No checkpoint found — run 03_Train_HAR_Head first.')
else:
    checkpoint = ckpts[-1]
    print(f'Using checkpoint: {checkpoint.name}')

    # Build clip samples for visualizations
    clips = resolve_training_clips(
        find_inhard_root(), exclude_labels=BLOCKED_ACTIONS, clips_per_class=5, seed=42
    )
    by_class = {}
    for rec in clips:
        by_class.setdefault(rec.label, []).append(rec.path)
    yolo_sample = [(rec.path, rec.label) for rec in clips[:16]]
    example_clip = clips[0].path if clips else None

In [ ]:
# Run full v2 analysis
result = run_v2_analysis(
    checkpoint,
    split                  = 'subject',
    sample_clips_for_strip = by_class,
    yolo_sample_clips      = yolo_sample,
    example_clip_for_3view = example_clip,
)
print(f"\nReport: {result['report_path']}")

In [ ]:
# Display all charts inline
charts = result['charts']
chart_titles = {
    'class_balance':     '1. Class balance',
    'confusion_matrix':  '2. Confusion matrix',
    'per_class_metrics': '3. Per-class P/R/F1',
    'training_history':  '4. Training history',
    'umap_clusters':     '5. UMAP clusters ← new',
    'frame_strip':       '6. Frame strip comparison ← new',
    'yolo_detection':    '7. YOLO detection grid ← new',
    'per_person_heatmap': '8. Per-person heatmap ← new',
    'confidence_calib':  '9. Confidence calibration ← new',
    'top_confusions':    '10. Top confusion pairs ← new',
    '3view_comparison':  '11. 3-view comparison ← new',
    'pca_outliers':      '13. PCA outliers',
}
for key, title in chart_titles.items():
    path = charts.get(key)
    if path and Path(path).is_file():
        display(Markdown(f'### {title}'))
        display(Image(path, width=820))
    else:
        print(f'  [{key}] not generated')

In [ ]:
# Print metrics summary
s = result['summary']
print(f"""
╔══════════════════════════════════════════╗
║  v2 HAR Analysis — {s['model_tag']:<22s}║
╠══════════════════════════════════════════╣
║  Accuracy      : {s['accuracy']:.1%:<26s}║
║  Macro F1      : {s['macro_f1']:.3f:<26s}║
║  Weighted F1   : {s['weighted_f1']:.3f:<26s}║
║  Macro Prec.   : {s['macro_precision']:.3f:<26s}║
║  Macro Recall  : {s['macro_recall']:.3f:<26s}║
║  Train clips   : {s['n_train']:<26d}║
║  Val clips     : {s['n_test']:<26d}║
╚══════════════════════════════════════════╝
""")

## DINOv2 Analysis (if checkpoint exists)

In [ ]:
dino_ckpts = sorted(CHECKPOINTS_DIR.glob('har_dinov2_*.pt'))
if dino_ckpts:
    dino_ckpt = dino_ckpts[-1]
    dino_npz  = OUTPUTS_DIR / 'embeddings_dinov2.npz'
    print(f'DINOv2 checkpoint: {dino_ckpt.name}')
    dino_result = run_v2_analysis(
        dino_ckpt, npz_path=dino_npz,
        split='subject',
        sample_clips_for_strip=by_class,
        yolo_sample_clips=yolo_sample,
    )
    ds = dino_result['summary']
    print(f"DINOv2 → acc={ds['accuracy']:.1%}  macro_f1={ds['macro_f1']:.3f}")
else:
    print('No DINOv2 checkpoint found.')

## Error Analysis — Worst Classes Deep Dive

In [ ]:
import pandas as pd

# Show worst F1 classes and their main confusors
eval_r    = result['eval']
cls_names = eval_r['class_names']
report    = eval_r['report']
cm        = eval_r['confusion_matrix']

rows = [(n, report[n]['f1-score'], report[n]['support']) for n in cls_names if n in report]
df   = pd.DataFrame(rows, columns=['class','F1','support']).sort_values('F1')
print('\nWorst 5 classes by F1:')
display(df.head(5))

# For each worst class, show top confusor
print('\nMain confusors for worst classes:')
for _, row in df.head(5).iterrows():
    true_idx = cls_names.index(row['class'])
    confused  = cm[true_idx].copy()
    confused[true_idx] = 0
    if confused.sum() == 0:
        continue
    top_pred  = confused.argmax()
    print(f"  {row['class']:30s} F1={row['F1']:.2f}  → confused with '{cls_names[top_pred]}' ({confused[top_pred]}x)")